# Build a SLM Lab

In [23]:
# Instantiate the Google Cloud Logging client
import logging

# Create a specific logger for cloud logging
logger = logging.getLogger('my_cloud_logger')

# Prevent this logger from sending messages to its parent (the root logger),
# which has the default console handler
logger.propagate = False

# Set the logging level
logger.setLevel(logging.INFO)

# Task 1. Define helper functions and load data

In [2]:
# Packages used
from collections import Counter, defaultdict
import random
import re
from typing import Any, Literal
import itertools

import keras
import numpy as np
import pandas as pd

In [ ]:
# Utils functions
def clean_bidi_chars(text: str) -> str:
   bidi_chars = re.compile(r"[\u202A-\u202E\u2066-\u2069]")
   return bidi_chars.sub("", text)


def remove_diacritics(text: str) -> str:
   arabic_diacritics = re.compile(r"[\u064B-\u065F\u0670\u0640]")
   return arabic_diacritics.sub("", text)


def display_arabic(text: str) -> str:
   return "\u202E" + text + "\u202C"


sentence = "ذَهَبَ الطَالبُ الى المَدْرَسَة"  # The student went to school
sentence_bidi_clean = clean_bidi_chars(sentence)
sentence_without_diacritics = remove_diacritics(sentence_bidi_clean)

print("EXAMPLE")
print(f"Text:           {sentence}")
print(f"Processed text: {display_arabic(sentence_without_diacritics)}")

EXAMPLE
Text:           ذَهَبَ الطَالبُ الى المَدْرَسَة
Processed text: ‮ذهب الطالب الى المدرسة‬


In [9]:
# Load the dataset
url = "https://storage.googleapis.com/dm-educational/assets/ai_foundations/200-stories-ar.json"

df = pd.read_json(url)

# Extract data & process the text
dataset = []
for story in df['story_ar'].to_list():
    story = clean_bidi_chars(story)
    story = remove_diacritics(story)
    dataset.append(story)

In [6]:
df.head()

,story_ar,story,id
0,الشمس طلعت! بطة صغيرة تستيقظ. تحب اللعب. ترى م...,The sun is up! A little duck wakes up. He like...,a5bee6d9-a351-4d70-a4d4-ffb721e3ac77
1,ليلي لديها دمية. الدمية ناعمة جداً. تحب ليلي أ...,Lily has a doll. The doll is very soft. Lily l...,7b1ecb85-cd53-4e3e-b750-366975f65adf
2,ليلي الخروفة أحبت الزهور. أحبت الزهور الصفراء ...,Lily the lamb loved flowers. She loved yellow ...,e8fde931-80a8-4fb4-82f7-a7b8b7bca3eb
3,كان لدى ليلي مجلد أزرق. كان مجلدًا مميزًا جدًا...,Lily had a blue folder. It was a very special ...,31f62512-6649-4d86-b5c4-7999e7435a54
4,كانت ليلي تحب اللعب في الخارج. كانت الشمس دافئ...,Lily loved to play outside. The sun was warm a...,66a8aea1-c824-4015-91bf-3176bcb93686


In [20]:
dataset[0]

'الشمس طلعت! بطة صغيرة تستيقظ. تحب اللعب. ترى مزارعا قرب بركتها. المزارع لطيف. يلوح للبطة.\n\nالبطة تتهادى إلى صديقها، أرنب ناعم. الأرنب يأكل جزرة. تقول البطة: "مرحبا! هل تريد اللعب؟" الأرنب يومئ برأسه. هما سعيدان.\n\nيريان صخرة ملساء ومسطحة. تبدو مكانا ممتعا للتزلج! يحاول الأرنب التزلج لكنه يقفز بدلا من ذلك. البطة تنزلق قليلا. من الممتع المحاولة!\n\nقريبا، حان وقت الغداء. البطة تأكل بذورا لذيذة. الأرنب يأكل المزيد من الجزر. يأخذان قيلولة في الشمس الدافئة. هما صديقان حميمان.'

# Task 2. Character Tokenizer

In [ ]:
class SimpleArabicCharacterTokenizer:

    # Constructor
    def __init__(self):
        pass

    # Split a give Arabic text into character tokens
    def character_tokenize(self, text: str) -> list[str]:
        # My code
        tokens = list(text)
        return tokens

    # Combine a list of token to a single string
    def join_text(self, tokens: list[str]) -> str:
        # My code
        text = "".join(tokens)
        return text

In [24]:
# Test your tokenizer on the first line of the first story from the dataset.

# Create tokenizer.
tokenizer = SimpleArabicCharacterTokenizer()

# Get first line of first story.
first_story = dataset[0]
first_line = first_story.split("\n")[0]
print(f"First line of dataset:\n\t{first_line}")

# Tokenize the line of text.
first_line_tokens = tokenizer.character_tokenize(first_line)
print(f"First line tokens:\n\t{first_line_tokens}")

# Join the tokens to reform the line of text.
first_line_rejoined =  tokenizer.join_text(first_line_tokens)
print(f"First line rejoined:\n\t{first_line_rejoined}")

# Do not remove or modify this logging call, it will be used for tracking purposes
logger.info(f'Task 2: First line rejoined: {len(first_line_tokens)}')

First line of dataset:
	الشمس طلعت! بطة صغيرة تستيقظ. تحب اللعب. ترى مزارعا قرب بركتها. المزارع لطيف. يلوح للبطة.
First line tokens:
	['ا', 'ل', 'ش', 'م', 'س', ' ', 'ط', 'ل', 'ع', 'ت', '!', ' ', 'ب', 'ط', 'ة', ' ', 'ص', 'غ', 'ي', 'ر', 'ة', ' ', 'ت', 'س', 'ت', 'ي', 'ق', 'ظ', '.', ' ', 'ت', 'ح', 'ب', ' ', 'ا', 'ل', 'ل', 'ع', 'ب', '.', ' ', 'ت', 'ر', 'ى', ' ', 'م', 'ز', 'ا', 'ر', 'ع', 'ا', ' ', 'ق', 'ر', 'ب', ' ', 'ب', 'ر', 'ك', 'ت', 'ه', 'ا', '.', ' ', 'ا', 'ل', 'م', 'ز', 'ا', 'ر', 'ع', ' ', 'ل', 'ط', 'ي', 'ف', '.', ' ', 'ي', 'ل', 'و', 'ح', ' ', 'ل', 'ل', 'ب', 'ط', 'ة', '.']
First line rejoined:
	الشمس طلعت! بطة صغيرة تستيقظ. تحب اللعب. ترى مزارعا قرب بركتها. المزارع لطيف. يلوح للبطة.


# Task 3. Generating text from an n-gram model

In [ ]:
# Generates character n-grams from a given text
def genrate_character_ngrams(text: str, n: int, tokenizer: SimpleArabicCharacterTokenizer) -> list[tuple[str]]:

    # tokenize the text
    tokens = tokenizer.character_tokenize(text)

    # Construct the list of n-grams
    ngrams = []

    for i in range(len(tokens) - n + 1):
        ngrams.append(tuple(tokens[i:n+i]))

    return ngrams

In [41]:
# Computes the character n-gram counts from a dataset
def get_character_ngram_counts(dataset: list[str], n: int, tokenizer: SimpleArabicCharacterTokenizer) -> dict[str, Counter]:
    
    ngrams_counts = defaultdict(Counter)

    # Constrcut ngrams for each text
    for paragraph in dataset:

        # Loop on each paragrapg in or to retrieve the ngrams
        for ngram in genrate_character_ngrams(paragraph, n, tokenizer):

            # Retrieve the context
            context = "".join(ngram[:n-1])

            # Retrieve the next token
            next_token = ngram[-1]

            # Increment the conter if next_token arrive after context
            ngrams_counts[context][next_token] += 1

    return ngrams_counts

In [42]:
# Builds an n-gram language model
def build_ngram_model(dataset: list[str], n: int, tokenizer: SimpleArabicCharacterTokenizer) -> dict[str, dict[str, float]]:

    # Build n-gram model
    ngram_model = {}

    # Retrieve each context and the tokens that follow it
    for context_key, next_tokens_counter in get_character_ngram_counts(dataset, n, tokenizer).items():
        ngram_model[context_key] = {}

        # Compute the total number of tokens following this context
        context_sum = sum(next_tokens_counter.values())

        # Retrieve each next token and its count
        for next_token, count in next_tokens_counter.items():

            # Compute the conditional probability
            ngram_model[context_key][next_token] = count / context_sum

    return ngram_model

In [43]:
def argmax(arr: list[float]) -> int:
    """Get the index of the largest element in list of float elements."""
    return max(range(len(arr)), key=arr.__getitem__)

In [44]:
# Generate text based on a starting prompt using an ngram model
def generate_text_from_ngram_model(
        start_prompt: str, n_tokens: int,
        ngram_model: dict[str, dict[str, float]], 
        tokenizer: SimpleArabicCharacterTokenizer, 
        sampling_mode: Literal["random", "greedy"] = "random") -> str:

    # Tokenize the starting prompt
    start_tokens = tokenizer.character_tokenize(start_prompt)

    # Generated tokens
    generated_tokens = start_tokens + []

    # My code
    # 1. Generate an additional `n_tokens` tokens iteratively, updating the context that is used for the n-gram model at each iteration.

    # Retrieve the nature of the ngram
    n = len(list(ngram_model.keys())[0]) + 1

    # Loop in order to generate next n_tokens tokens
    for i in range(n_tokens):

        # Updating the context
        context = generated_tokens[-(n - 1):]

        # List of potential next tokens and their probability
        potential_next_tokens = []
        potential_next_token_probility = []

        # 2. Use the n-gram model to provide the conditional probabilities of the next token given the context.
        for token, probability in ngram_model["".join(context)].items():
            potential_next_tokens.append(token)
            potential_next_token_probility.append(probability)
            
        # 3. Support "random" or "greedy" sampling depending on `sampling_mode`.
        #    - Random sampling must use the `random.choice` function to sample the next token to add to the generated text.
        if (sampling_mode == "greedy"):
            generated_tokens.append(potential_next_tokens[argmax(potential_next_token_probility)])

        #    - Greedy sampling can use the `argmax` function to pick the token that has highest probability as the next token.
        else:
            generated_tokens.append(random.choice(potential_next_tokens))

    # 4. Return the complete resulting sequence as a `str` object. This should include the text from the starting prompt.
    generated_text = tokenizer.join_text(generated_tokens)
    return generated_text

In [45]:
# Train n-gram model from dataset.
n = 4 # Size of n-grams.
ngram_model = build_ngram_model(dataset, n, tokenizer)

# Generate text.
start_prompt = "يوم واحد"
print(f"Start prompt is:\n\t{start_prompt}")
n_tokens = 15 # Specify the number of new tokens to generate.
tokenizer = SimpleArabicCharacterTokenizer()

generated_text = generate_text_from_ngram_model(
    start_prompt,
    n_tokens,
    ngram_model,
    tokenizer,
    sampling_mode="random")

print(f"Text generated is:\n\t{display_arabic(generated_text)}")

# Do not remove or modify this logging call, it will be used for tracking purposes
logger.info(f'Task 3: The total word count for the generated text is: {len(generated_text.split())}')

Start prompt is:
	يوم واحد
Text generated is:
	‮يوم واحدا قضيا شروق يضح‬
